#### Import libraries

In [17]:
!pip install xgboost

In [18]:
import pandas as pd
import numpy as np 

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

import xgboost as xgb

In [19]:
df = pd.read_csv('car_fuel_efficiency_2026.csv')

df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [20]:
df.shape

(10000, 11)

In [21]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

#### Data Cleaning

In [22]:
df = df.fillna(0)

In [23]:
df.isnull().sum()

model_year             0
origin                 0
fuel_type              0
drivetrain             0
num_doors              0
engine_displacement    0
num_cylinders          0
horsepower             0
vehicle_weight         0
acceleration           0
fuel_efficiency_mpg    0
dtype: int64

In [24]:
# splitting data 60% train, 20% validation, 20% test
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

print(len(df_train))
print(len(df_val))
print(len(df_test))

6000
2000
2000


In [ ]:
# seperate the target from features
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values

# removing target columns from feature tables
df_train = df_train.drop(columns=['fuel_efficiency_mpg'])
df_val = df_val.drop(columns=['fuel_efficiency_mpg'])

In [26]:
# changing each row into a dict
train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

dv = DictVectorizer(sparse=True)


X_train = dv.fit_transform(train_dicts)

X_val = dv.transform(val_dicts)

print(dv.get_feature_names_out())

['acceleration' 'drivetrain=All-wheel drive'
 'drivetrain=Front-wheel drive' 'drivetrain=Rear-wheel drive'
 'engine_displacement' 'fuel_type=Diesel' 'fuel_type=Gasoline'
 'fuel_type=Hybrid' 'horsepower' 'model_year' 'num_cylinders' 'num_doors'
 'origin=Asia' 'origin=Europe' 'origin=USA' 'vehicle_weight']


#### Decision tree with ```max_depth = 1```
Which original feature is used for splitting the data?

In [30]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

feature_number = dt.tree_.feature[0]
feature_name = dv.get_feature_names_out()[feature_number]
print('Feature used (after DictVectorizer):', feature_name)

original_column = feature_name.split('=')[0]
print('Original column:', original_column)

Feature used (after DictVectorizer): model_year
Original column: model_year


#### Random forest with ```n_estimators=10```
What's the RMSE of this model on the validation data?

In [31]:
# random forest model train
rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)

rmse = np.sqrt(mean_squared_error(y_val, y_pred))
print('RMSE:', round(rmse,4))

RMSE: 1.8371


#### Trying defferent ```n_estimators```
Which value gives the lowest validation RMSE?

In [32]:
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    y_pred = rf.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred))
    print('n_estimators=', n, 'with RMSE =', round(rmse, 3))

n_estimators= 10 with RMSE = 1.837
n_estimators= 50 with RMSE = 1.772
n_estimators= 100 with RMSE = 1.768
n_estimators= 150 with RMSE = 1.769


#### Best ```max_depth```
trying all 4 ```n_estimators``` for each max depth, then take the mean rmse

In [34]:
# dict to store rmse for each max_depth
mean_rmse = {}

for depth in [10, 15, 20, 25]:

    # list to store rmse for each depth
    rmse_list = []

    # loop for each n_extimiators
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=depth, random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        y_pred = rf.predict(X_val)
        rmse = np.sqrt(mean_squared_error(y_val, y_pred))
        rmse_list.append(rmse)

    
    mean_rmse[depth] = np.mean(rmse_list)
    print('max_depth =', depth, 'with mean rmse', round(mean_rmse[depth], 4))

best_depth = min(mean_rmse, key=mean_rmse.get)
print('Best max_depth:', best_depth)


max_depth = 10 with mean rmse 1.7554
max_depth = 15 with mean rmse 1.784
max_depth = 20 with mean rmse 1.7882
max_depth = 25 with mean rmse 1.7863
Best max_depth: 10


#### Feature importance
What's the most important feature among said selections ?

here, i try to train with ```n_estimator=10```, ```max_depth=20``` and find the most important feature on said selection

In [36]:
# Train model with given parameters 
rf = RandomForestRegressor(n_estimators=10, max_depth=20, random_state=1, n_jobs=-1)
rf.fit(X_train,y_train)

importances = pd.DataFrame({
    'feature': dv.get_feature_names_out(),
    'importance': rf.feature_importances_
})

importances.sort_values('importance', ascending=False)

,feature,importance
9,model_year,0.296001
15,vehicle_weight,0.203879
6,fuel_type=Gasoline,0.160712
8,horsepower,0.078017
4,engine_displacement,0.061005
0,acceleration,0.055201
14,origin=USA,0.042144
2,drivetrain=Front-wheel drive,0.020145
11,num_doors,0.019415
5,fuel_type=Diesel,0.017038


In [38]:
# seeing the said 4 features
features = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
importances[importances['feature'].isin(features)].sort_values('importance', ascending=False)

,feature,importance
15,vehicle_weight,0.203879
8,horsepower,0.078017
4,engine_displacement,0.061005
0,acceleration,0.055201


#### XGBoost model and ```eta``` hyperparameter tuning

comparing ```eta=0.3``` and ```eta=0.1``` hyperparameter tuning over 100 rounds. 

In [40]:
features = list(dv.get_feature_names_out())
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=features)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=features)

watchlist = [(dtrain, 'train'), (dval,'val')]

In [ ]:
# Parameters for xgb, first with eta 0.3
xgb_params = {
    'eta': 0.3,
    'max_depth': 6,
    'min_child_weight': 1,
    'objective': 'reg:squarederror',
    'nthread': 8,
    'seed': 1,
    'verbosity': 1,
}

results_03 = {}

# Training xgb model for 100 rounds
model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist, evals_result=results_03, verbose_eval=False)

# RMSE Validation after last train
val_scores_03 = results_03['val']['rmse']
print('eta=0.3 with last rmse', round(val_scores_03[-1],4), '| best rmse:', round(min(val_scores_03), 4))


eta=0.3 with last rmse 1.8265 | best rmse: 1.732


In [43]:
# another train with xgb model with eta 0.1
xgb_params['eta'] = 0.1

results_01 = {}
model = xgb.train(xgb_params, dtrain, num_boost_round=100, evals=watchlist, evals_result=results_01, verbose_eval=False)

val_scores_01 = results_01['val']['rmse']
print('eta=0.1 with last rmse:', round(val_scores_01[-1],4), '| best rmse:', round(min(val_scores_01), 4))


eta=0.1 with last rmse: 1.7248 | best rmse: 1.7122


In [44]:
# comparing two xgb models with different eta

print('eta=0.3: best rmse =', round(min(val_scores_03), 3))
print('eta=0.1: best rmse =', round(min(val_scores_01), 3))

if round(min(val_scores_03), 3) < round(min(val_scores_01), 3):
    print('Best: 0.3')
elif round(min(val_scores_03), 3) > round(min(val_scores_01), 3):
    print('Best: 0.1')
else:
    print('Both give equal value')

eta=0.3: best rmse = 1.732
eta=0.1: best rmse = 1.712
Best: 0.1
